# AgriShift — 03. Feature Engineering (Revised)

**Project:** Agricultural Diversification & Crop Pattern Intelligence in India  
**SDG:** 2 — Zero Hunger

## Purpose
Convert cleaned source records into district-year crop portfolios and diversification indicators.

## Why this notebook was revised
The source contains aggregate labels (`Total foodgrain`, `Pulses total`, `Oilseeds total`) and alternative labels for some of the same crop categories (`Paddy`/`Rice`, raw/processed cashew, raw/processed arecanut). Treating these as separate crops can double-count area and distort diversity scores.

This notebook excludes the three aggregate labels from derived portfolios, standardizes the identified aliases, audits duplicate keys, and retains one preferred source row for each duplicate alias key.

## Important definition
We sum `Area` across recorded seasons for the same district-year-crop before calculating shares. The resulting shares describe a category's proportion of **total reported crop area across recorded seasons**, not the proportion of a district's unique physical land. The same land may be cultivated more than once in a year.

Residual groups such as `Other Kharif pulses` remain as reported categories because the source does not provide a reliable breakdown. Our output therefore represents reported crop categories, not a perfect count of botanical crop species.

## Outputs
- `crop_portfolio_records.csv`
- `district_year_diversification_features.csv`
- `district_year_crop_share_matrix.csv`
- `district_year_season_share_matrix.csv`
- `crop_label_audit.csv`

The raw and cleaned source files are not modified.

## 1. Import libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 150)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

## 2. Locate and load the cleaned dataset

In [ ]:
CLEANED_RELATIVE_PATH = Path("data") / "processed" / "crop_production_cleaned.csv"

PROJECT_ROOT = next(
    (
        folder
        for folder in (Path.cwd(), *Path.cwd().parents)
        if (folder / CLEANED_RELATIVE_PATH).is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find data/processed/crop_production_cleaned.csv. "
        "Run Notebook 02 and verify the processed folder name."
    )

CLEANED_PATH = PROJECT_ROOT / CLEANED_RELATIVE_PATH
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CLEANED_PATH)

required_columns = [
    "State_Name", "District_Name", "Crop_Year",
    "Season", "Crop", "Area", "Production"
]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print(f"Loaded: {CLEANED_PATH}")
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
display(df.head())

## 3. Standardize labels and exclude aggregate totals

We remove whitespace defensively and exclude the three summary labels from the **derived portfolio only**. They remain untouched in the cleaned source CSV.

In [ ]:
df_work = df.copy()

for column in ["State_Name", "District_Name", "Season", "Crop"]:
    df_work[column] = (
        df_work[column].astype("string")
        .str.replace(r"\\s+", " ", regex=True)
        .str.strip()
    )

aggregate_labels = {"Total foodgrain", "Pulses total", "Oilseeds total"}
aggregate_mask = df_work["Crop"].isin(aggregate_labels)

aggregate_audit = (
    df_work.loc[aggregate_mask, "Crop"]
    .value_counts()
    .rename_axis("source_label")
    .reset_index(name="records_excluded")
)

print(f"Aggregate-total rows excluded from portfolio: {int(aggregate_mask.sum()):,}")
display(aggregate_audit)

df_work = df_work.loc[~aggregate_mask].copy()

## 4. Normalize known crop aliases

| Source label | Standardized label |
|---|---|
| Paddy | Rice |
| Cashewnut Raw | Cashewnut |
| Cashewnut Processed | Cashewnut |
| Arcanut (Processed) | Arecanut |
| Atcanut (Raw) | Arecanut |

These mappings stop known alternate labels from being treated as separate crop categories.

In [ ]:
alias_map = {
    "Paddy": "Rice",
    "Cashewnut Raw": "Cashewnut",
    "Cashewnut Processed": "Cashewnut",
    "Arcanut (Processed)": "Arecanut",
    "Atcanut (Raw)": "Arecanut",
}

df_work["Source_Crop_Label"] = df_work["Crop"]
df_work["Crop"] = df_work["Crop"].replace(alias_map)

alias_audit = (
    df_work.loc[df_work["Source_Crop_Label"].isin(alias_map), "Source_Crop_Label"]
    .value_counts()
    .rename_axis("source_label")
    .reset_index(name="records_mapped")
)
alias_audit["standardized_label"] = alias_audit["source_label"].map(alias_map)

display(alias_audit)
print(f"Standardized crop categories before duplicate collapse: {df_work['Crop'].nunique()}")

## 5. Audit duplicate keys created by alias normalization

After mapping, multiple source rows may share the same state, district, year, season and standardized crop. We must not sum duplicate area rows.

The dataset audit identified duplicate alias groups. In most groups, area values are identical. If a group has a differing area value, the discrepancy is surfaced for review; the standard crop label is preferred by the documented selection rule.

In [ ]:
record_key = ["State_Name", "District_Name", "Crop_Year", "Season", "Crop"]

duplicate_mask = df_work.duplicated(subset=record_key, keep=False)
duplicate_audit_rows = []

for group_key, group in df_work.loc[duplicate_mask].groupby(record_key, dropna=False):
    area_values = group["Area"].dropna().unique()
    duplicate_audit_rows.append({
        "State_Name": group_key[0],
        "District_Name": group_key[1],
        "Crop_Year": group_key[2],
        "Season": group_key[3],
        "standardized_crop": group_key[4],
        "source_labels": " | ".join(sorted(group["Source_Crop_Label"].astype(str).unique())),
        "source_rows": len(group),
        "area_values": " | ".join(map(str, sorted(area_values))),
        "area_conflict": len(area_values) > 1,
    })

duplicate_audit = pd.DataFrame(duplicate_audit_rows)

print(f"Duplicate canonical groups: {len(duplicate_audit):,}")
if not duplicate_audit.empty:
    print(f"Groups with different area values: {int(duplicate_audit['area_conflict'].sum())}")
    display(duplicate_audit.loc[duplicate_audit["area_conflict"]])
else:
    print("No duplicate canonical groups found.")

## 6. Collapse duplicate aliases using a documented priority

When duplicate aliases refer to the same district-season-year-crop, retain one row rather than adding their areas.

Priority: standard crop name first, raw variant next, processed variant last. The standard `Rice` row is preferred over `Paddy` where they map to the same key.

In [ ]:
source_priority = {
    "Rice": 0,
    "Cashewnut": 0,
    "Arecanut": 0,
    "Paddy": 1,
    "Cashewnut Raw": 1,
    "Atcanut (Raw)": 1,
    "Cashewnut Processed": 2,
    "Arcanut (Processed)": 2,
}

df_work["_source_priority"] = (
    df_work["Source_Crop_Label"].map(source_priority).fillna(0)
)

rows_before_deduplication = len(df_work)

portfolio_records = (
    df_work.sort_values("_source_priority", kind="stable")
    .drop_duplicates(subset=record_key, keep="first")
    .drop(columns="_source_priority")
    .copy()
)

removed_alias_rows = rows_before_deduplication - len(portfolio_records)

print(f"Rows before duplicate-alias collapse: {rows_before_deduplication:,}")
print(f"Duplicate alias rows removed: {removed_alias_rows:,}")
print(f"Portfolio records retained: {len(portfolio_records):,}")
print(f"Remaining duplicate keys: {portfolio_records.duplicated(record_key).sum()}")

assert portfolio_records.duplicated(record_key).sum() == 0

## 7. Validate area and aggregate by district, year and crop

In [ ]:
if portfolio_records["Area"].isna().any() or portfolio_records["Area"].lt(0).any():
    raise ValueError("Area contains missing or negative values; inspect before continuing.")

DISTRICT_YEAR_KEYS = ["State_Name", "District_Name", "Crop_Year"]

crop_area = (
    portfolio_records
    .groupby(DISTRICT_YEAR_KEYS + ["Crop"], as_index=False)
    .agg(
        crop_area_ha=("Area", "sum"),
        seasons_recorded=("Season", "nunique"),
        source_rows=("Crop", "size"),
    )
)

print(f"District-year-crop records: {len(crop_area):,}")
print(f"Standardized crop categories: {crop_area['Crop'].nunique()}")
display(crop_area.head(10))

## 8. Calculate crop-area shares

In [ ]:
crop_area["total_reported_crop_area_ha"] = (
    crop_area.groupby(DISTRICT_YEAR_KEYS)["crop_area_ha"].transform("sum")
)

if crop_area["total_reported_crop_area_ha"].le(0).any():
    raise ValueError("A district-year has zero or negative total reported crop area.")

crop_area["crop_area_share"] = (
    crop_area["crop_area_ha"] / crop_area["total_reported_crop_area_ha"]
)

share_sum_check = crop_area.groupby(DISTRICT_YEAR_KEYS)["crop_area_share"].sum()

print("Minimum share sum:", share_sum_check.min())
print("Maximum share sum:", share_sum_check.max())
print("All district-year shares sum to 1:",
      bool(np.allclose(share_sum_check.to_numpy(), 1.0)))
display(crop_area.head(10))

## 9. Calculate crop diversification and concentration indicators

- **Crop richness:** number of standardized crop categories recorded.
- **Shannon entropy:** reflects category count and how evenly area is distributed.
- **Normalized Shannon evenness:** Shannon entropy divided by maximum possible entropy; ranges from 0 to 1.
- **HHI concentration:** sum of squared crop shares; higher values mean greater concentration.
- **Simpson diversity:** 1 minus HHI; higher values mean less concentration.
- **Effective number of crops:** exponential of Shannon entropy.
- **Top-one and top-three shares:** share of reported crop area represented by the largest one or three categories.
- **Dominant crop category:** category with the largest area share.

A district-year with only one reported crop category gets HHI = 1. This may reflect incomplete reporting rather than real specialization, so we flag it for later review.

In [ ]:
feature_rows = []

for (state, district, year), group in crop_area.groupby(DISTRICT_YEAR_KEYS, sort=False):
    ranked = group.sort_values("crop_area_share", ascending=False)
    shares = ranked["crop_area_share"].to_numpy(dtype=float)
    shares = shares[shares > 0]

    crop_count = len(shares)
    entropy = float(-(shares * np.log(shares)).sum())
    hhi = float(np.square(shares).sum())
    evenness = entropy / np.log(crop_count) if crop_count > 1 else 0.0

    feature_rows.append({
        "State_Name": state,
        "District_Name": district,
        "Crop_Year": int(year),
        "total_reported_crop_area_ha": float(ranked["total_reported_crop_area_ha"].iloc[0]),
        "crop_richness": int(crop_count),
        "shannon_entropy": entropy,
        "normalized_shannon_evenness": float(evenness),
        "hhi_concentration": hhi,
        "simpson_diversity": 1.0 - hhi,
        "effective_number_of_crops": float(np.exp(entropy)),
        "top_1_crop_share": float(shares[0]),
        "top_3_crop_share": float(shares[:3].sum()),
        "dominant_crop": str(ranked["Crop"].iloc[0]),
        "crop_category_records": int(len(group)),
        "single_crop_observation": bool(crop_count == 1),
    })

crop_diversity_features = pd.DataFrame(feature_rows)

print("District-year feature table:", crop_diversity_features.shape)
display(crop_diversity_features.head())

## 10. Calculate seasonal distribution features

In [ ]:
season_area = (
    portfolio_records
    .groupby(DISTRICT_YEAR_KEYS + ["Season"], as_index=False)
    .agg(season_area_ha=("Area", "sum"))
)

season_area["total_reported_crop_area_ha"] = (
    season_area.groupby(DISTRICT_YEAR_KEYS)["season_area_ha"].transform("sum")
)
season_area["season_area_share"] = (
    season_area["season_area_ha"] / season_area["total_reported_crop_area_ha"]
)

season_feature_rows = []

for (state, district, year), group in season_area.groupby(DISTRICT_YEAR_KEYS, sort=False):
    ranked = group.sort_values("season_area_share", ascending=False)
    shares = ranked["season_area_share"].to_numpy(dtype=float)
    shares = shares[shares > 0]

    season_count = len(shares)
    entropy = float(-(shares * np.log(shares)).sum())
    hhi = float(np.square(shares).sum())
    evenness = entropy / np.log(season_count) if season_count > 1 else 0.0

    season_feature_rows.append({
        "State_Name": state,
        "District_Name": district,
        "Crop_Year": int(year),
        "season_richness": int(season_count),
        "season_shannon_entropy": entropy,
        "normalized_season_evenness": float(evenness),
        "season_hhi_concentration": hhi,
        "dominant_season": str(ranked["Season"].iloc[0]),
    })

season_features = pd.DataFrame(season_feature_rows)

season_share_matrix = (
    season_area.pivot_table(
        index=DISTRICT_YEAR_KEYS,
        columns="Season",
        values="season_area_share",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
)
season_share_matrix.columns.name = None

print("Season feature table:", season_features.shape)
print("Season share matrix:", season_share_matrix.shape)
display(season_features.head())

## 11. Create the district-year crop-share matrix

In [ ]:
crop_share_matrix = (
    crop_area.pivot_table(
        index=DISTRICT_YEAR_KEYS,
        columns="Crop",
        values="crop_area_share",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
)
crop_share_matrix.columns.name = None

crop_columns = [
    column for column in crop_share_matrix.columns
    if column not in DISTRICT_YEAR_KEYS
]
row_share_sums = crop_share_matrix[crop_columns].sum(axis=1)

print("Crop-share matrix shape:", crop_share_matrix.shape)
print("Crop-category features:", len(crop_columns))
print("Minimum row share sum:", row_share_sums.min())
print("Maximum row share sum:", row_share_sums.max())
print("All rows sum to 1:", bool(np.allclose(row_share_sums.to_numpy(), 1.0)))
display(crop_share_matrix.head())

## 12. Add the common district-name panel indicator

For selected historical comparisons, mark exact `(State_Name, District_Name)` pairs that occur in every year from 1997 through 2014. This is a name-based panel only; it cannot prove district boundaries remained unchanged.

In [ ]:
comparison_years = sorted(
    year for year in df["Crop_Year"].dropna().unique()
    if 1997 <= year <= 2014
)

district_sets_by_year = []
for year in comparison_years:
    pairs = (
        df.loc[df["Crop_Year"].eq(year), ["State_Name", "District_Name"]]
        .drop_duplicates()
    )
    district_sets_by_year.append(set(map(tuple, pairs.to_numpy())))

common_districts = (
    set.intersection(*district_sets_by_year)
    if district_sets_by_year else set()
)

diversification_features = crop_diversity_features.merge(
    season_features,
    on=DISTRICT_YEAR_KEYS,
    how="left",
    validate="one_to_one",
)

diversification_features["in_common_name_panel_1997_2014"] = [
    (state, district) in common_districts and 1997 <= year <= 2014
    for state, district, year in diversification_features[
        DISTRICT_YEAR_KEYS
    ].itertuples(index=False, name=None)
]

print("Common state-district name pairs:", len(common_districts))
print(
    "District-year observations in common name-based panel:",
    int(diversification_features["in_common_name_panel_1997_2014"].sum())
)
print("Combined feature table shape:", diversification_features.shape)
display(diversification_features.head())

## 13. Validate all engineered features

In [ ]:
expected_district_years = (
    df[DISTRICT_YEAR_KEYS].drop_duplicates().shape[0]
)

assert len(diversification_features) == expected_district_years
assert len(crop_share_matrix) == expected_district_years
assert np.allclose(row_share_sums.to_numpy(), 1.0)

assert crop_share_matrix[crop_columns].ge(0).all().all()
assert crop_share_matrix[crop_columns].le(1).all().all()

for column in [
    "normalized_shannon_evenness",
    "hhi_concentration",
    "simpson_diversity",
    "normalized_season_evenness",
]:
    assert diversification_features[column].between(0, 1).all()

print("Feature validation passed.")
print(f"District-year observations: {expected_district_years:,}")
print(f"Crop-category share columns: {len(crop_columns)}")
print(
    "Single-crop observations:",
    int(diversification_features["single_crop_observation"].sum())
)
print(
    "Common name-based panel observations:",
    int(diversification_features["in_common_name_panel_1997_2014"].sum())
)

## 14. Inspect selected indicators

In [ ]:
display(
    diversification_features[
        [
            "State_Name", "District_Name", "Crop_Year",
            "crop_richness", "normalized_shannon_evenness",
            "hhi_concentration", "effective_number_of_crops",
            "top_1_crop_share", "top_3_crop_share",
            "dominant_crop", "dominant_season",
            "single_crop_observation",
        ]
    ].head(15)
)

print("\\nMost concentrated district-year portfolios:")
display(
    diversification_features.sort_values("hhi_concentration", ascending=False)[
        [
            "State_Name", "District_Name", "Crop_Year",
            "crop_richness", "hhi_concentration",
            "top_1_crop_share", "dominant_crop",
            "single_crop_observation",
        ]
    ].head(10)
)

## 15. Save derived datasets and crop-label audit

In [ ]:
portfolio_records_path = PROCESSED_DIR / "crop_portfolio_records.csv"
features_path = PROCESSED_DIR / "district_year_diversification_features.csv"
crop_matrix_path = PROCESSED_DIR / "district_year_crop_share_matrix.csv"
season_matrix_path = PROCESSED_DIR / "district_year_season_share_matrix.csv"

portfolio_records.to_csv(portfolio_records_path, index=False)
diversification_features.to_csv(features_path, index=False)
crop_share_matrix.to_csv(crop_matrix_path, index=False)
season_share_matrix.to_csv(season_matrix_path, index=False)

aggregate_audit_out = aggregate_audit.copy()
aggregate_audit_out["audit_type"] = "aggregate_label_excluded"
aggregate_audit_out["standardized_label"] = ""

alias_audit_out = alias_audit.copy()
alias_audit_out["audit_type"] = "alias_standardized"
alias_audit_out["records_excluded"] = 0

duplicate_audit_out = duplicate_audit.copy()
duplicate_audit_out["audit_type"] = "duplicate_alias_group"
duplicate_audit_out["records_excluded"] = 1
duplicate_audit_out["standardized_label"] = duplicate_audit_out["standardized_crop"]

crop_label_audit = pd.concat(
    [aggregate_audit_out, alias_audit_out, duplicate_audit_out],
    ignore_index=True,
    sort=False,
)
audit_path = PROCESSED_DIR / "crop_label_audit.csv"
crop_label_audit.to_csv(audit_path, index=False)

for path in [
    portfolio_records_path,
    features_path,
    crop_matrix_path,
    season_matrix_path,
    audit_path,
]:
    print(f"{path.name}: {path.stat().st_size / (1024 * 1024):.2f} MB")

## 16. Reopen and verify saved outputs

In [ ]:
features_check = pd.read_csv(features_path)
crop_matrix_check = pd.read_csv(crop_matrix_path)
season_matrix_check = pd.read_csv(season_matrix_path)
portfolio_check = pd.read_csv(portfolio_records_path)

assert features_check.shape == diversification_features.shape
assert crop_matrix_check.shape == crop_share_matrix.shape
assert season_matrix_check.shape == season_share_matrix.shape
assert portfolio_check.shape == portfolio_records.shape

assert np.allclose(
    crop_matrix_check[crop_columns].sum(axis=1).to_numpy(),
    1.0,
)

print("All derived files were saved and read back successfully.")
print("Portfolio records:", portfolio_check.shape)
print("Diversification features:", features_check.shape)
print("Crop-share matrix:", crop_matrix_check.shape)
print("Season-share matrix:", season_matrix_check.shape)

## 17. Interpretation and limitations

### How to interpret indicators
- Higher `crop_richness` means more standardized crop categories are recorded.
- Higher `normalized_shannon_evenness` means reported crop area is distributed more evenly across categories present.
- Higher `hhi_concentration` means reported crop area is concentrated in fewer categories.
- Higher `simpson_diversity` means less concentration.
- `effective_number_of_crops` translates Shannon entropy into the equivalent number of equally represented categories.
- `single_crop_observation` flags district-years with only one recorded category. This may reflect incomplete reporting and must not automatically be called true specialization.

### Important limitations
1. Shares use reported crop area summed across seasons, not unique physical district land.
2. Three aggregate-total labels were excluded from the derived portfolio to avoid overlap with component crop records.
3. Known aliases were standardized. Ambiguous residual categories remain as reported categories.
4. Geographic coverage varies by year.
5. The 1997–2014 panel uses exact state and district names and cannot guarantee stable administrative boundaries.
6. These are descriptive indicators; they do not identify causes.

### Stage outcome
The cleaned source has been transformed into a documented crop portfolio with alias handling, aggregate-label exclusion, district-year diversification indicators, and crop/season share matrices. These outputs are ready for EDA and later clustering.